# Mask the release model's unlabelled crowns in HOT train

Run on a GPU runtime (L4). Set `REPO_SHA` in the next cell, then run the cells in order.

The release model (the teacher) runs over every HOT train region at `WINDOW_M` and keeps its detections scoring at least `SCORE`. A detection that overlaps no HOT label becomes an ignore square (`reason: "teacher"`), so fine-tuning is not punished for finding trees the mappers did not map. Two shard sets are written: `shards/hot-teacher-all` masks such detections everywhere, `shards/hot-teacher-empty` only on tiles no label touches. Val regions are copied unchanged, so val scores compare with runs on `shards/hot`.

The repo is public; this notebook still clones it with a read-only GitHub token from Colab Secrets (the key icon), named `GITHUB_TOKEN`, with notebook access on.

In [ ]:
REPO_SHA = ""
WINDOW_M = 60.0  # the release model's best window on HOT val
SCORE = 0.3  # teacher detections below this score are not masked

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install "/content/fair_trees[train,test]" -r /content/fair_trees/experiments/requirements.txt
%cd /content/fair_trees
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import json

import torch

from experiments import run
from experiments.shards import stage_shards

print(torch.cuda.get_device_name())
stage_shards(DRIVE, Path("/content/data"), ["hot"])

In [ ]:
import logging
import shutil

from experiments.teacher import SCOPES, write_teacher_sets
from fair_trees.params import InferenceParams

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
OUT = Path("/content/teacher")
# scratch on the Colab VM only, so a rerun after a disconnect starts clean
shutil.rmtree(OUT, ignore_errors=True)
stats = write_teacher_sets(
    Path("/content/data/hot"),
    OUT,
    run._run_fn(DRIVE / "weights" / "NEON.pt", None),
    InferenceParams(window_m=WINDOW_M, confidence_threshold=SCORE),
)
print(json.dumps(stats, indent=2))

In [ ]:
from experiments.shards import manifest_hash, write_shards

for name in SCOPES.values():
    shard_dir = DRIVE / "shards" / name
    assert not (shard_dir / "manifest.json").exists(), f"{shard_dir} exists: move it aside first"
    write_shards(OUT / name, shard_dir, source=name, converter_sha=REPO_SHA)
    shutil.copyfile(OUT / name / "teacher.json", shard_dir / "teacher.json")
    print(name, "manifest sha256:", manifest_hash(shard_dir / "manifest.json"))

In [ ]:
import random

from affine import Affine
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageDraw

from fair_trees.dataset import labels_to_boxes, load_geojson
from fair_trees.georef import tile_bounds_3857
from fair_trees.preprocess import CHIP_SUFFIXES, read_chip

THUMB = 256
masked_root = OUT / SCOPES["all"]
train_dirs = [
    p.parent for p in masked_root.glob("*/region.json") if json.loads(p.read_text())["split"] == "train"
]
chip_paths = sorted(p for d in train_dirs for p in d.glob("z*/chips/*") if p.suffix.lower() in CHIP_SUFFIXES)
picked = random.Random(0).sample(chip_paths, min(16, len(chip_paths)))
contact = PILImage.new("RGB", (4 * THUMB, 4 * THUMB))
for i, path in enumerate(picked):
    rgb, _, (x, y, z) = read_chip(path, use_sidecar=False)
    minx, miny, maxx, maxy = tile_bounds_3857(x, y, z)
    res = (maxx - minx) / rgb.shape[1]
    transform = Affine(res, 0, minx, 0, -res, maxy)
    region = path.parents[2]
    tile = PILImage.fromarray(rgb[..., :3])
    draw = ImageDraw.Draw(tile)
    ignore = (
        load_geojson(region / "ignore.geojson") if (region / "ignore.geojson").is_file() else {"features": []}
    )
    teacher = [f for f in ignore["features"] if f["properties"].get("reason") == "teacher"]
    other = [f for f in ignore["features"] if f["properties"].get("reason") != "teacher"]
    # HOT labels red, teacher masks blue, other ignore (groups, unknown diameters) grey
    layers = (
        ({"features": other}, (128, 128, 128)),
        ({"features": teacher}, (0, 96, 255)),
        (load_geojson(region / "labels.geojson"), (255, 0, 0)),
    )
    for collection, colour in layers:
        for x0, y0, x1, y1 in labels_to_boxes(collection, transform):
            draw.rectangle((x0, y0, x1, y1), outline=colour, width=2)
    contact.paste(tile.resize((THUMB, THUMB)), ((i % 4) * THUMB, (i // 4) * THUMB))
sheet = DRIVE / "shards" / SCOPES["all"] / "qa-teacher-contact-sheet.png"
contact.save(sheet)
display(Image(filename=str(sheet)))

In [ ]:
drive.flush_and_unmount()